# M3 · Ejecución integral del sistema RAG clínico

Une el trabajo de **todo el equipo** para el Módulo 3 y corre el pipeline de
punta a punta:

```
corpus (Isa)  →  retrieval + orquestación (Pau)  →  generación (Agustín)  →  RAGAS + cruce con M2 (Luis)
```

**Funciona igual en Google Colab y en local.** El notebook detecta el entorno y
se adapta: en Colab clona la rama `dev` y monta Drive; en local usa el repositorio
donde ya está y un directorio de datos propio.

Las celdas son Python puro (sin `%pip`, `!` ni `%cd`), así que corren en cualquier
kernel (Colab, Jupyter, VS Code) y también se pueden ejecutar como script.

**Requisitos:** `GROQ_API_KEY` (generación y RAGAS) y, opcionalmente,
`BIOPORTAL_API_KEY` (tool de normalización real). En Colab van en Secretos; en
local en el entorno o en un `.env`.

## 1 · Entorno y parámetros

Detecta Colab vs. local y define las rutas. En local se puede sobrescribir con
`M3_REPO_DIR` (raíz del repo) y `M3_DATA_ROOT` (datos), si no se toman los valores
por defecto.

In [1]:
import os, sys, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False


def _raiz_repo(inicio):
    p = Path(inicio).resolve()
    for cand in [p, *p.parents]:
        if (cand / "M3").is_dir() and (cand / ".git").exists():
            return cand
    return p


RAMA = "dev"
REPO_URL = "https://github.com/luisNP21/Topicos-IA.git"

if EN_COLAB:
    REPO_DIR = Path("/content/Topicos-IA")
    DATA_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M3")
else:
    REPO_DIR = _raiz_repo(os.environ.get("M3_REPO_DIR") or os.getcwd())
    DATA_ROOT = Path(os.environ.get("M3_DATA_ROOT") or (REPO_DIR.parent / "m3_data")).resolve()

# Eval set del equipo (15 casos: entidad + pregunta + esperado).
EJEMPLOS = "M3/ejecucion/eval_set_casos.json"
EJEMPLOS_ALT = "M3/generacion/entidades_ejemplo.json"

MODO_RAGAS = "real"      # "real" (librería ragas + juez Groq) | "mock"
CORRER_CORPUS = True     # False si el índice ya está construido
CORRER_EXPERIMENTO = True  # calibra umbrales y mide el delta del retrieval

print("Entorno:", "Colab" if EN_COLAB else "local")
print("REPO_DIR:", REPO_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("RAGAS:", MODO_RAGAS, "| corpus:", CORRER_CORPUS, "| experimento:", CORRER_EXPERIMENTO)

Entorno: local
REPO_DIR: D:\Clase-topicos-IA\Dupla\proyecto
DATA_ROOT: D:\Clase-topicos-IA\Dupla\m3_local_run
RAGAS: real | corpus: True | experimento: True


## 2 · Sincronizar el repositorio

En **Colab** se clona/sincroniza `dev` (el runtime se pierde al cerrar la sesión).
En **local** se usa el repositorio tal cual, **sin** clonar ni resetear (no se
pisan cambios locales).

In [2]:
def ejecutar(cmd, cwd=None, extra_env=None, check=True):
    """Corre un comando mostrando su salida; aborta si falla."""
    env = {**os.environ, **(extra_env or {})}
    print("$", " ".join(str(c) for c in cmd))
    r = subprocess.run([str(c) for c in cmd], cwd=str(cwd) if cwd else None,
                       env=env, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout)
    if r.stderr:
        print(r.stderr)
    if check and r.returncode:
        raise RuntimeError("Fallo (%d): %s" % (r.returncode, " ".join(str(c) for c in cmd)))
    return r


if EN_COLAB:
    if (REPO_DIR / ".git").exists():
        ejecutar(["git", "-C", REPO_DIR, "fetch", "origin", RAMA])
        ejecutar(["git", "-C", REPO_DIR, "checkout", "-B", RAMA, "origin/" + RAMA])
        ejecutar(["git", "-C", REPO_DIR, "reset", "--hard", "origin/" + RAMA])
    else:
        ejecutar(["git", "clone", "--branch", RAMA, REPO_URL, REPO_DIR])
    print(ejecutar(["git", "-C", REPO_DIR, "log", "-1", "--oneline"]).stdout.strip())
else:
    print("Modo local: se usa el repo en", REPO_DIR, "(no se clona ni se resetea).")

Modo local: se usa el repo en D:\Clase-topicos-IA\Dupla\proyecto (no se clona ni se resetea).


## 3 · Instalar dependencias

Cada pieza trae su `requirements.txt`; el retrieval no, así que sus dependencias
se instalan aparte.

In [3]:
for req in ["M3/corpus", "M3/tools", "M3/generacion", "M3/ragas"]:
    ejecutar([sys.executable, "-m", "pip", "install", "-q", "-r", REPO_DIR / req / "requirements.txt"])

ejecutar([sys.executable, "-m", "pip", "install", "-q",
          "sentence-transformers", "chromadb", "rank_bm25", "groq", "python-dotenv", "PyYAML"])
print("Dependencias instaladas.")

$ C:\venvs\topicos-ia\Scripts\python.exe -m pip install -q -r D:\Clase-topicos-IA\Dupla\proyecto\M3\corpus\requirements.txt
$ C:\venvs\topicos-ia\Scripts\python.exe -m pip install -q -r D:\Clase-topicos-IA\Dupla\proyecto\M3\tools\requirements.txt
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
docling-core 2.99.0 requires requests<3.0.0,>=2.34.2, but you have requests 2.32.3 which is incompatible.
kubernetes 36.0.3 requires pyyaml>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
langchain-community 0.3.31 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.3 which is incompatible.

$ C:\venvs\topicos-ia\Scripts\python.exe -m pip install -q -r D:\Clase-topicos-IA\Dupla\proyecto\M3\generacion\requirements.txt
$ C:\venvs\topicos-ia\Scripts\python.exe -m pip install -q -r D:\Clase-topicos-IA\Dupla\proyecto\M3\ragas\requirements.txt
ERROR: pip's dep

## 4 · Datos y credenciales

- **Colab:** monta Drive y toma las claves de los Secretos.
- **Local:** toma las claves del entorno o de un `.env`.

En ambos casos se escribe un `.env` en la raíz del repo (ignorado por git) para
que los entry points lo lean.

In [4]:
if EN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    for var in ("GROQ_API_KEY", "BIOPORTAL_API_KEY"):
        try:
            os.environ[var] = userdata.get(var)
        except Exception:
            pass
else:
    try:
        from dotenv import load_dotenv
        for p in [REPO_DIR / ".env", REPO_DIR / "M3/generacion/.env", REPO_DIR / "M2/harness/.env"]:
            if p.exists():
                load_dotenv(p, override=False)
    except Exception as e:
        print("Aviso cargando .env:", e)

lineas = ["PROJECT_ROOT=" + str(REPO_DIR)]
for var in ("GROQ_API_KEY", "BIOPORTAL_API_KEY"):
    if os.environ.get(var):
        lineas.append(var + "=" + os.environ[var])
(REPO_DIR / ".env").write_text("\n".join(lineas) + "\n", encoding="utf-8")
print("Claves presentes:", [v for v in ("GROQ_API_KEY", "BIOPORTAL_API_KEY") if os.environ.get(v)])
print(".env escrito en", REPO_DIR / ".env")

Claves presentes: ['GROQ_API_KEY', 'BIOPORTAL_API_KEY']
.env escrito en D:\Clase-topicos-IA\Dupla\proyecto\.env


## 5 · Configuración del retrieval según el entorno

El YAML del repo (`config_retrieval.yaml`) apunta a Drive para Colab. En **local**
se arma un YAML equivalente (sin tocar el del repo) con el perfil `real` apuntando
a `DATA_ROOT`, y se exporta `M3_RETRIEVAL_CONFIG` para que los módulos lo usen.

In [5]:
import yaml

os.environ["M3_REPO_ROOT"] = str(REPO_DIR)
os.environ["M3_DATA_ROOT"] = str(DATA_ROOT)

# Sin BIOPORTAL_API_KEY, la tool real de Luis cae a un mock diminuto que no conoce
# las siglas del eval set (DM2, HTA, EPOC...). En ese caso se usa el mapa simulado
# del equipo para que la normalizacion funcione en la corrida de demostracion.
mapa_mock = None if os.environ.get("BIOPORTAL_API_KEY") else "M3/retrieval/data/mock/normalizacion_mock.json"


def _cfg_retrieval():
    return yaml.safe_load((REPO_DIR / "M3/retrieval/config_retrieval.yaml").read_text(encoding="utf-8"))


if EN_COLAB:
    if mapa_mock:
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        cfg_r = _cfg_retrieval()
        cfg_r["rutas"]["real"]["mapa_normalizacion"] = mapa_mock
        tmp = DATA_ROOT / "config_retrieval.colab.yaml"
        tmp.write_text(yaml.safe_dump(cfg_r, allow_unicode=True, sort_keys=False), encoding="utf-8")
        os.environ["M3_RETRIEVAL_CONFIG"] = str(tmp)
        print("Sin BIOPORTAL: se usa el mapa de normalizacion simulado.")
    else:
        print("Colab: se usa M3/retrieval/config_retrieval.yaml (rutas de Drive).")
else:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    cfg_r = _cfg_retrieval()
    base = str(DATA_ROOT)
    cfg_r["rutas"]["real"].update({
        "corpus_json": base + "/data/guias_clinicas/chunks",
        "chroma_origen": base + "/data/chroma_guias",
        "chroma_dir": base + "/chroma_local",
        "manifest": base + "/data/guias_clinicas/corpus_manifest.json",
        "salida": "M3/retrieval/outputs",
    })
    if mapa_mock:
        cfg_r["rutas"]["real"]["mapa_normalizacion"] = mapa_mock
    cfg_r["rutas"]["mock"]["chroma_dir"] = base + "/chroma_mock"
    local_cfg = DATA_ROOT / "config_retrieval.local.yaml"
    local_cfg.write_text(yaml.safe_dump(cfg_r, allow_unicode=True, sort_keys=False), encoding="utf-8")
    os.environ["M3_RETRIEVAL_CONFIG"] = str(local_cfg)
    print("YAML de retrieval local:", local_cfg)



YAML de retrieval local: D:\Clase-topicos-IA\Dupla\m3_local_run\config_retrieval.local.yaml


## 6 · Verificar recursos

Comprueba las guías (PDF), el resultado de M2 para el cruce y el eval set.

In [6]:
guia_dir = DATA_ROOT / "data/guias_clinicas"
checks = {
    "Guías PDF (data/guias_clinicas)": guia_dir if guia_dir.exists() else (REPO_DIR / "M3/corpus/guias_clinicas"),
    "Resultado M2 (para el cruce)": REPO_DIR / "M2/ejecucion/outputs_gold/resultado_dimension1.json",
    "Eval set del equipo": REPO_DIR / EJEMPLOS,
}
for nombre, p in checks.items():
    print("%s: %s -- %s" % (nombre, "OK" if p.exists() else "FALTA", p))

ejemplos = REPO_DIR / EJEMPLOS
if ejemplos.exists():
    ENTIDADES = EJEMPLOS
elif (REPO_DIR / EJEMPLOS_ALT).exists():
    ENTIDADES = EJEMPLOS_ALT
    print("Aviso: no existe %s; se usa %s" % (EJEMPLOS, EJEMPLOS_ALT))
else:
    raise FileNotFoundError("No hay eval set de entidades.")

Guías PDF (data/guias_clinicas): OK -- D:\Clase-topicos-IA\Dupla\m3_local_run\data\guias_clinicas
Resultado M2 (para el cruce): OK -- D:\Clase-topicos-IA\Dupla\proyecto\M2\ejecucion\outputs_gold\resultado_dimension1.json
Eval set del equipo: OK -- D:\Clase-topicos-IA\Dupla\proyecto\M3\ejecucion\eval_set_casos.json


## 7 · Pruebas unitarias

Rápidas y sin descargar modelos. (Algunas pruebas de retrieval dependen del perfil
`corpus` activo en el YAML; pueden fallar si el YAML trae `real`.)

In [7]:
ejecutar([sys.executable, "-m", "unittest", "test_retrieval"], cwd=REPO_DIR / "M3/retrieval", check=False)
ejecutar([sys.executable, "-m", "unittest", "test_generacion"], cwd=REPO_DIR / "M3/generacion", check=False)

$ C:\venvs\topicos-ia\Scripts\python.exe -m unittest test_retrieval
Aviso: no hay umbrales calibrados en C:\Users\AGUSTN~1\AppData\Local\Temp\tmpmn0e_s37\umbrales.json; se usan los provisionales del YAML.

...................................
----------------------------------------------------------------------
Ran 35 tests in 0.117s

OK

$ C:\venvs\topicos-ia\Scripts\python.exe -m unittest test_generacion
[00:18:48] Piezas REALES: retrieval (M3/retrieval/config_retrieval.yaml) + normalizacion (corpus 'mock').
    [orquestacion] falló el retrieval ('neumonia'): indice caido
    [orquestacion] falló el retrieval ('ENFERMEDAD CONTROLADA'): indice caido

................
----------------------------------------------------------------------
Ran 16 tests in 0.035s

OK



## 8 · Corpus (Isa): ingesta de las guías + índice

Ingesta → chunking → embeddings → Chroma + manifest. Es **idempotente**: las
fuentes ya indexadas se saltan por caché de `doc_id`.

In [8]:
if CORRER_CORPUS:
    import shutil
    destino = DATA_ROOT / "data/guias_clinicas"
    destino.mkdir(parents=True, exist_ok=True)
    for pdf in sorted((REPO_DIR / "M3/corpus/guias_clinicas").glob("*.pdf")):
        if not (destino / pdf.name).exists():
            shutil.copy2(pdf, destino / pdf.name)
            print("copiada:", pdf.name)
    ejecutar([sys.executable, "run_corpus.py", "--config", "config.yaml", "--fuentes", "fuentes.yaml"],
             cwd=REPO_DIR / "M3/corpus", extra_env={"PROJECT_ROOT": str(DATA_ROOT)})
else:
    print("CORRER_CORPUS=False: se usa el índice ya construido.")

$ C:\venvs\topicos-ia\Scripts\python.exe run_corpus.py --config config.yaml --fuentes fuentes.yaml
CACHE gpc_diabetes_mellitus_tipo_2: chunks ya existen en disco, se omite reprocesamiento
CACHE gpc_falla_cardiaca: chunks ya existen en disco, se omite reprocesamiento
CACHE gpc_depresion_adultos: chunks ya existen en disco, se omite reprocesamiento
{'n_fuentes_ok': 3, 'n_fuentes_cacheadas': 3, 'n_fuentes_nuevas': 0, 'n_fuentes_fallidas': 0, 'n_documentos': 3, 'n_chunks': 64, 'n_secciones_filtradas': 0, 'errores': []}

Skipping import of cpp extensions due to incompatible torch version 2.14.0+cpu for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7472.13it/s]



## 9 · Retrieval (Pau): calibración de umbrales y delta

Experimento controlado (ingenuo vs. híbrido vs. reranker vs. sistema completo).
Escribe `umbrales.json` y `deltas_s08.csv` en `M3/retrieval/outputs/`.

In [9]:
if CORRER_EXPERIMENTO:
    r = ejecutar([sys.executable, "experimento_s08.py", "--corpus", "real"],
                 cwd=REPO_DIR / "M3/retrieval", check=False)
    if r.returncode:
        print("\nAVISO: el experimento de calibracion fallo "
              "(las consultas del corpus real pueden estar sin etiquetar). "
              "Se continua con los umbrales provisionales del YAML.")
else:
    print("CORRER_EXPERIMENTO=False: se usan los umbrales provisionales del YAML.")


$ C:\venvs\topicos-ia\Scripts\python.exe experimento_s08.py --corpus real
[tool_normalizacion] descartado: 'sepsis' -> 'Sepsis' (sin aporte)
[tool_normalizacion] descartado: 'anemia' -> 'Anemia' (sin aporte)
[tool_normalizacion] descartado: 'SIADH' -> 'Syndrome of inappropriate vasopressin secretion' (sin relacion)
[tool_normalizacion] descartado: 'WPW' -> 'Wolff-Parkinson-White pattern' (sin relacion)
[tool_normalizacion] descartado: 'GIST' -> 'Benign gastrointestinal stromal tumor of stomach' (sin relacion)
[tool_normalizacion] descartado: 'IR aguda' -> 'ph IR' (sin relacion)
[tool_normalizacion] descartado: 'insuficiencia respiratoria refractaria secundaria a síndrome de distrés respiratorio agudo' -> 'Hemoglobin A,a' (sin relacion)
[tool_normalizacion] descartado: 'tumor' -> 'Neoplasm' (sin relacion)
[tool_normalizacion] descartado: 'DM2' -> 'dm2' (sin aporte)
[tool_normalizacion] descartado: 'sepsis' -> 'Sepsis' (sin aporte)
[tool_normalizacion] descartado: 'anemia' -> 'Anemia' (s

## 10 · Generación (Agustín): respuesta final del RAG

Encadena orquestación (cuándo se invoca la tool y qué hace si falla) + generación.
Consume el retrieval de Pau y la tool de Luis desde sus YAML y escribe
`M3/outputs/resultado_generacion.json`, que consume RAGAS.

In [10]:
ejecutar([sys.executable, "run_generacion.py", "--config", "config.yaml", "--entidades", ENTIDADES],
         cwd=REPO_DIR / "M3/generacion", extra_env={"PROJECT_ROOT": str(REPO_DIR)})

$ C:\venvs\topicos-ia\Scripts\python.exe run_generacion.py --config config.yaml --entidades M3/ejecucion/eval_set_casos.json
[00:33:34] PROJECT_ROOT: D:\Clase-topicos-IA\Dupla\proyecto
[00:33:34] Entidades: 15 (desde M3/ejecucion/eval_set_casos.json)
[00:33:57] Piezas REALES: retrieval (M3/retrieval/config_retrieval.yaml) + normalizacion (corpus 'real').
[00:33:57] Generando sobre 15 entidades (umbral=0.176584, umbral_evidencia=0.147845, k=5, modelo=qwen/qwen3.8-27b)...
[00:34:02]   [1/15] 'TFNA' (abreviatura) -> query='TFNA' | tool=True | fragmentos=0 | fallback=True | fuentes=0
[tool_normalizacion] descartado: 'neumonía por broncoaspiración' -> 'Insertion of preventive resin tooth restoration' (sin relacion)
[00:34:06]   [2/15] 'neumonía por broncoaspiración' (multimorbilidad) -> query='neumonía por broncoaspiración' | tool=True | fragmentos=0 | fallback=True | fuentes=0
[00:34:11]   [3/15] 'artritis séptica' (urgencia) -> query='artritis séptica' | tool=True | fragmentos=0 | fallbac

## 11 · RAGAS + cruce con el harness de M2 (Luis)

Faithfulness, context precision, context recall y answer relevancy, cruzados con
el F1 de extracción de M2 para diagnosticar dónde falla el sistema.

In [11]:
ejecutar([sys.executable, "run_ragas.py", "--config", "config.yaml", "--modo", MODO_RAGAS],
         cwd=REPO_DIR / "M3/ragas", extra_env={"PROJECT_ROOT": str(REPO_DIR)})

$ C:\venvs\topicos-ia\Scripts\python.exe run_ragas.py --config config.yaml --modo real
[ragas] Cargados 15 casos reales desde D:\Clase-topicos-IA\Dupla\proyecto\M3\outputs\resultado_generacion.json
[ragas] Modo: 'real' | Fuente de datos: D:\Clase-topicos-IA\Dupla\proyecto\M3\outputs\resultado_generacion.json | Casos: 15

--- Metricas RAGAS calculadas ---
  Faithfulness (Fidelidad)     : 0.3571
  Context Precision (Precision): 0.0000
  Context Recall (Cobertura)   : 0.0000
  Answer Relevancy (Relevancia): 0.0000

TABLA DE CRUCE RAGAS x HARNESS M2
| doc_id | F1 extraccion (M2) | Context recall | Faithfulness | Diagnostico |
|---|---|---|---|---|
| ex_0 | 0.000 | 0.000 | 0.357 | alucinacion_generacion |
| ex_1 | 0.000 | 0.000 | 0.357 | alucinacion_generacion |
| ex_2 | 0.667 | 0.000 | 0.357 | problema_corpus_retrieval |
| ex_3 | 0.667 | 0.000 | 0.357 | problema_corpus_retrieval |
| ex_4 | 0.400 | 0.000 | 0.357 | problema_corpus_retrieval |
| ex_5 | 0.500 | 0.000 | 0.357 | problema_corpus_

## 12 · Persistir los artefactos

Copia los resultados a `DATA_ROOT/resultados_pipeline` y arma un zip. En Colab
también ofrece la descarga.

In [12]:
import shutil

persist = DATA_ROOT / "resultados_pipeline"
persist.mkdir(parents=True, exist_ok=True)
for origen, nombre in [(REPO_DIR / "M3/outputs", "generacion"),
                       (REPO_DIR / "M3/retrieval/outputs", "retrieval")]:
    if origen.exists():
        shutil.copytree(origen, persist / nombre, dirs_exist_ok=True)
        print("copiado:", persist / nombre)

zip_path = shutil.make_archive(str(DATA_ROOT / "resultados_M3"), "zip", str(persist))
print("zip:", zip_path)

if EN_COLAB:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print("Descarga manual en:", zip_path, "|", e)

copiado: D:\Clase-topicos-IA\Dupla\m3_local_run\resultados_pipeline\generacion
copiado: D:\Clase-topicos-IA\Dupla\m3_local_run\resultados_pipeline\retrieval
zip: D:\Clase-topicos-IA\Dupla\m3_local_run\resultados_M3.zip
